# Geometry-Aware Drone Deployment: Experiment Notebook

This notebook runs the full simulation pipeline and visualizes results:
1. **Antenna radiation patterns** (downlink + backhaul)
2. **Channel model validation** (LoS probability and path loss curves)
3. **Single scenario analysis** (3D plot, SINR heat map, CDF, interference graph)
4. **Placement strategy comparison** (K-means vs grid vs random)
5. **Tilt sensitivity analysis** (the core thesis insight: antenna orientation matters)

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt

from dronecomm.antenna import ParametricAntenna, DOWNLINK_ANTENNA, BACKHAUL_ANTENNA
from dronecomm.channel import ChannelModel, SUBURBAN, URBAN, DENSE_URBAN
from dronecomm.scenario import create_scenario
from dronecomm.interference import downlink_power_matrix, backhaul_interference_matrix
from dronecomm.sinr import compute_sinr, nearest_drone_association, dbm_to_watts
from dronecomm.metrics import compute_metrics
from dronecomm.config import Config
from dronecomm.simulation import run_trial, run_monte_carlo
from dronecomm.visualization import (
    plot_scenario_3d,
    plot_antenna_pattern_polar,
    plot_sinr_heatmap,
    plot_sinr_cdf,
    plot_interference_graph,
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "font.size": 11})
print("Imports OK")

---
## 1. Antenna Radiation Patterns

The parametric model follows the 3GPP TR 38.901 single-element pattern:

$$G(\theta) = G_{\max} - \min\!\left(12\left(\frac{\theta}{\theta_{3\text{dB}}}\right)^{\!2},\; \text{SLA}\right) \quad [\text{dBi}]$$

Each drone carries two antennas:
- **Downlink** (wide beam, nadir): serves ground users
- **Backhaul** (narrow beam, horizontal): drone-to-drone links

In [ ]:
fig, axes = plt.subplots(1, 2, subplot_kw={"projection": "polar"}, figsize=(14, 6))

for ax, antenna, name in [
    (axes[0], DOWNLINK_ANTENNA, "Downlink"),
    (axes[1], BACKHAUL_ANTENNA, "Backhaul"),
]:
    theta = np.linspace(0, np.pi, 500)
    gain = antenna.gain_dbi(theta)
    ax.plot(theta, gain, linewidth=2, color="darkblue")
    ax.plot(-theta, gain, linewidth=2, color="darkblue")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_rlabel_position(135)
    ax.set_title(
        f"{name} Antenna\n"
        f"$G_{{\\max}}$={antenna.g_max_dbi} dBi, "
        f"$\\theta_{{3dB}}$={antenna.beamwidth_deg}\u00b0, "
        f"SLA={antenna.sla_db} dB",
        pad=20, fontsize=11,
    )

plt.tight_layout()
plt.show()

In [ ]:
# Gain vs off-boresight angle (Cartesian view)
theta_deg = np.linspace(0, 180, 500)
theta_rad = np.deg2rad(theta_deg)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(theta_deg, DOWNLINK_ANTENNA.gain_dbi(theta_rad), linewidth=2, label="Downlink (60\u00b0, 18 dBi)")
ax.plot(theta_deg, BACKHAUL_ANTENNA.gain_dbi(theta_rad), linewidth=2, label="Backhaul (30\u00b0, 12 dBi)")

# Mark the -3 dB points
for ant, color in [(DOWNLINK_ANTENNA, "C0"), (BACKHAUL_ANTENNA, "C1")]:
    half_bw = ant.beamwidth_deg / 2
    g_3db = ant.g_max_dbi - 3
    ax.plot(half_bw, g_3db, "o", color=color, markersize=8)
    ax.annotate(f"-3 dB at {half_bw}\u00b0", (half_bw + 2, g_3db + 0.5), color=color, fontsize=9)

ax.set_xlabel("Off-boresight angle [deg]")
ax.set_ylabel("Gain [dBi]")
ax.set_title("Antenna Gain vs Off-Boresight Angle")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Channel Model Validation

The Al-Hourani / ITU-R probabilistic LoS model:

$$P_{\text{LoS}}(\theta) = \frac{1}{1 + a \cdot \exp\!\left(-b\,(\theta_{\text{deg}} - a)\right)}$$

And the total path loss:

$$\text{PL} = \text{FSPL}(d, f_c) + P_{\text{LoS}} \cdot \eta_{\text{LoS}} + (1 - P_{\text{LoS}}) \cdot \eta_{\text{NLoS}}$$

In [ ]:
theta_deg = np.linspace(0, 90, 200)
theta_rad = np.deg2rad(theta_deg)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LoS probability vs elevation angle
ax = axes[0]
for env, label, color in [
    (SUBURBAN, "Suburban", "green"),
    (URBAN, "Urban", "orange"),
    (DENSE_URBAN, "Dense Urban", "red"),
]:
    ch = ChannelModel(env=env)
    p_los = ch.los_probability(theta_rad)
    ax.plot(theta_deg, p_los, linewidth=2, label=label, color=color)

ax.set_xlabel("Elevation angle [deg]")
ax.set_ylabel("P(LoS)")
ax.set_title("LoS Probability vs Elevation Angle")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Path loss vs horizontal distance (at 150 m altitude)
ax = axes[1]
altitude = 150.0
horiz_dist = np.linspace(50, 2000, 200)
d_3d = np.sqrt(horiz_dist**2 + altitude**2)
elev = np.arctan2(altitude, horiz_dist)

for env, label, color in [
    (SUBURBAN, "Suburban", "green"),
    (URBAN, "Urban", "orange"),
    (DENSE_URBAN, "Dense Urban", "red"),
]:
    ch = ChannelModel(env=env)
    pl = ch.path_loss_db(d_3d, elev)
    ax.plot(horiz_dist, pl, linewidth=2, label=label, color=color)

# Free-space reference
ch_ref = ChannelModel(env=URBAN)
ax.plot(horiz_dist, ch_ref.fspl_db(d_3d), "k--", linewidth=1, alpha=0.5, label="FSPL only")

ax.set_xlabel("Horizontal distance [m]")
ax.set_ylabel("Path loss [dB]")
ax.set_title(f"Path Loss vs Distance (altitude = {altitude} m, f = 2 GHz)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

---
## 3. Single Scenario Analysis

Generate a disaster scenario with 15 drones and ~200 users, then visualize the deployment, SINR coverage, and interference.

In [ ]:
# Generate scenario
scenario = create_scenario(
    n_drones=15,
    placement="kmeans",
    altitude_m=150.0,
    n_clusters=5,
    users_per_cluster_mean=40,
    cluster_spread_m=100.0,
    area_size_m=2000.0,
    seed=42,
)

n_users = scenario.user_positions.shape[0]
n_critical = int(np.sum(scenario.user_tiers == 1))
n_regular = int(np.sum(scenario.user_tiers == 0))
n_drones = scenario.drone_positions.shape[0]

print(f"Scenario: {n_drones} drones, {n_users} users ({n_critical} critical, {n_regular} regular)")
print(f"Area: {scenario.area_size_m} x {scenario.area_size_m} m")
print(f"Drone altitude: {scenario.drone_positions[0, 2]:.0f} m")

In [ ]:
# User-drone association
association = nearest_drone_association(scenario.drone_positions, scenario.user_positions)

# 3D scenario visualization
fig = plot_scenario_3d(
    scenario.drone_positions,
    scenario.user_positions,
    scenario.user_tiers,
    association,
    title=f"Disaster Scenario: {n_drones} Drones, {n_users} Users (K-Means Placement)",
)
plt.show()

In [ ]:
# Compute SINR for this scenario
channel = ChannelModel(env=URBAN)
p_tx_dl_w = dbm_to_watts(30.0)  # 1 W

power_mat = downlink_power_matrix(
    drone_positions=scenario.drone_positions,
    dl_tilt_rad=scenario.dl_tilt_rad,
    dl_azimuth_rad=scenario.dl_azimuth_rad,
    user_positions=scenario.user_positions,
    dl_antenna=DOWNLINK_ANTENNA,
    channel=channel,
    p_tx_dl_w=p_tx_dl_w,
)

sinr_result = compute_sinr(power_mat, association)

print(f"SINR statistics:")
print(f"  Mean:           {np.mean(sinr_result.sinr_db):.1f} dB")
print(f"  Median:         {np.median(sinr_result.sinr_db):.1f} dB")
print(f"  Min:            {np.min(sinr_result.sinr_db):.1f} dB")
print(f"  5th percentile: {np.percentile(sinr_result.sinr_db, 5):.1f} dB")
print(f"  Max:            {np.max(sinr_result.sinr_db):.1f} dB")
print(f"  Sum throughput: {np.sum(sinr_result.throughput_bps) / 1e6:.1f} Mbps")

In [ ]:
# SINR heat map on the ground plane
fig = plot_sinr_heatmap(
    drone_positions=scenario.drone_positions,
    dl_tilt_rad=scenario.dl_tilt_rad,
    dl_azimuth_rad=scenario.dl_azimuth_rad,
    dl_antenna=DOWNLINK_ANTENNA,
    channel=channel,
    p_tx_dl_w=p_tx_dl_w,
    area_size_m=scenario.area_size_m,
    grid_resolution=150,
    title="SINR Heat Map (K-Means, Nadir-Pointing, Urban)",
)
plt.show()

In [ ]:
# SINR CDF by user tier
fig = plot_sinr_cdf(
    sinr_result.sinr_db,
    scenario.user_tiers,
    sinr_critical_db=10.0,
    sinr_regular_db=3.0,
    title="SINR CDF by User Tier (K-Means Placement)",
)
plt.show()

In [ ]:
# Backhaul interference between drones
p_tx_bh_w = dbm_to_watts(27.0)

bh_interf = backhaul_interference_matrix(
    drone_positions=scenario.drone_positions,
    bh_tilt_rad=scenario.bh_tilt_rad,
    bh_azimuth_rad=scenario.bh_azimuth_rad,
    bh_antenna=BACKHAUL_ANTENNA,
    channel=channel,
    p_tx_bh_w=p_tx_bh_w,
)

fig = plot_interference_graph(
    scenario.drone_positions, bh_interf,
    title="Inter-Drone Backhaul Interference (Top-Down View)",
)
plt.show()

In [ ]:
# Full metrics summary
metrics = compute_metrics(
    sinr_result, scenario.user_tiers,
    sinr_critical_db=10.0, sinr_regular_db=3.0,
    backhaul_interference_matrix=bh_interf,
)

print("=== Single Scenario Metrics ===")
print(f"  Critical SINR satisfied:  {metrics.critical_sinr_satisfied:.1%}")
print(f"  Regular SINR satisfied:   {metrics.regular_sinr_satisfied:.1%}")
print(f"  Overall coverage:         {metrics.coverage_fraction:.1%}")
print(f"  Sum throughput:           {metrics.sum_throughput_mbps:.1f} Mbps")
print(f"  Mean SINR:               {metrics.mean_sinr_db:.1f} dB")
print(f"  5th %ile SINR:           {metrics.sinr_5th_percentile_db:.1f} dB")
print(f"  BH interference (total): {metrics.total_inter_drone_interference_dbm:.1f} dBm")
print(f"  BH interference (worst): {metrics.worst_pair_interference_dbm:.1f} dBm")

---
## 4. Placement Strategy Comparison

Compare the three baseline deployment strategies across 50 Monte Carlo trials each:
- **K-means**: positions above user cluster centroids
- **Grid**: regular grid covering the area
- **Random**: uniform random placement

In [ ]:
from dataclasses import replace

strategies = ["kmeans", "grid", "random"]
strategy_results = {}

base_config = Config()
base_config = replace(base_config, simulation=replace(base_config.simulation, n_trials=50))

for strategy in strategies:
    print(f"\nRunning {strategy}...")
    cfg = replace(base_config, simulation=replace(base_config.simulation, placement_strategy=strategy))
    result = run_monte_carlo(cfg, verbose=False)
    strategy_results[strategy] = result
    s = result.summary()
    print(f"  Coverage: {s['coverage_fraction']['mean']:.1%} +/- {s['coverage_fraction']['std']:.1%}")
    print(f"  Critical satisfied: {s['critical_sinr_satisfied']['mean']:.1%}")
    print(f"  Mean SINR: {s['mean_sinr_db']['mean']:.1f} dB")
    print(f"  Throughput: {s['sum_throughput_mbps']['mean']:.0f} Mbps")

print("\nDone.")

In [ ]:
# Bar chart comparison
metric_keys = [
    ("coverage_fraction", "Coverage Fraction", "{:.1%}"),
    ("critical_sinr_satisfied", "Critical Tier Satisfied", "{:.1%}"),
    ("mean_sinr_db", "Mean SINR [dB]", "{:.1f}"),
    ("sum_throughput_mbps", "Sum Throughput [Mbps]", "{:.0f}"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
x = np.arange(len(strategies))
colors = ["#2ecc71", "#3498db", "#e74c3c"]

for ax, (key, label, fmt) in zip(axes, metric_keys):
    means = [strategy_results[s].mean_metric(key) for s in strategies]
    stds = [strategy_results[s].std_metric(key) for s in strategies]
    bars = ax.bar(x, means, yerr=stds, color=colors, capsize=5, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([s.capitalize() for s in strategies])
    ax.set_title(label)
    ax.grid(axis="y", alpha=0.3)

    # Annotate bars
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                fmt.format(mean), ha="center", va="bottom", fontsize=9)

plt.suptitle("Placement Strategy Comparison (50 Monte Carlo Trials)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# SINR CDF overlay across strategies (using the first trial of each)
fig, ax = plt.subplots(figsize=(9, 5))
colors_line = {"kmeans": "#2ecc71", "grid": "#3498db", "random": "#e74c3c"}

for strategy in strategies:
    trial = strategy_results[strategy].trial_results[0]
    sinr = np.sort(trial.sinr_result.sinr_db)
    cdf = np.arange(1, len(sinr) + 1) / len(sinr)
    ax.step(sinr, cdf, where="post", linewidth=2,
            label=strategy.capitalize(), color=colors_line[strategy])

ax.axvline(10.0, color="red", linestyle="--", alpha=0.4, label="Critical thr (10 dB)")
ax.axvline(3.0, color="blue", linestyle="--", alpha=0.4, label="Regular thr (3 dB)")
ax.set_xlabel("SINR [dB]")
ax.set_ylabel("CDF")
ax.set_title("SINR CDF Comparison Across Placement Strategies")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. Tilt Sensitivity Analysis

This is the **core thesis insight**: tilting the downlink antenna away from nadir changes the interference pattern dramatically. We sweep the downlink tilt angle from 0\u00b0 (straight down) to 45\u00b0 and observe the effect on coverage, SINR, and interference.

In [ ]:
tilt_angles_deg = np.arange(0, 50, 5)  # 0 to 45 degrees
tilt_metrics = {"tilt_deg": [], "coverage": [], "mean_sinr": [],
                "critical_sat": [], "p5_sinr": [], "throughput": []}

# Use a fixed scenario (same users, same drone horizontal positions)
base_scenario = create_scenario(
    n_drones=15, placement="kmeans", altitude_m=150.0,
    n_clusters=5, users_per_cluster_mean=40,
    area_size_m=2000.0, critical_fraction=0.15, seed=42,
)
channel = ChannelModel(env=URBAN)
p_tx_w = dbm_to_watts(30.0)

for tilt_deg in tilt_angles_deg:
    tilt_rad = np.deg2rad(tilt_deg)
    dl_tilt = np.full(base_scenario.drone_positions.shape[0], tilt_rad)

    pm = downlink_power_matrix(
        base_scenario.drone_positions, dl_tilt, base_scenario.dl_azimuth_rad,
        base_scenario.user_positions, DOWNLINK_ANTENNA, channel, p_tx_w,
    )
    assoc = nearest_drone_association(base_scenario.drone_positions, base_scenario.user_positions)
    sinr = compute_sinr(pm, assoc)
    m = compute_metrics(sinr, base_scenario.user_tiers)

    tilt_metrics["tilt_deg"].append(tilt_deg)
    tilt_metrics["coverage"].append(m.coverage_fraction)
    tilt_metrics["mean_sinr"].append(m.mean_sinr_db)
    tilt_metrics["critical_sat"].append(m.critical_sinr_satisfied)
    tilt_metrics["p5_sinr"].append(m.sinr_5th_percentile_db)
    tilt_metrics["throughput"].append(m.sum_throughput_mbps)

print("Tilt sweep complete.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
x = tilt_metrics["tilt_deg"]

ax = axes[0, 0]
ax.plot(x, tilt_metrics["mean_sinr"], "o-", linewidth=2, color="#2c3e50")
ax.set_ylabel("Mean SINR [dB]")
ax.set_title("Mean SINR vs Downlink Tilt")
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(x, tilt_metrics["coverage"], "s-", linewidth=2, color="#27ae60")
ax.set_ylabel("Coverage Fraction")
ax.set_title("Coverage vs Downlink Tilt")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
ax.plot(x, tilt_metrics["critical_sat"], "D-", linewidth=2, color="#e74c3c")
ax.set_ylabel("Critical Tier Satisfied")
ax.set_title("Critical User Satisfaction vs Downlink Tilt")
ax.set_xlabel("Downlink tilt from nadir [deg]")
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
ax.plot(x, tilt_metrics["p5_sinr"], "^-", linewidth=2, color="#8e44ad")
ax.set_ylabel("5th Percentile SINR [dB]")
ax.set_title("Edge-User SINR vs Downlink Tilt")
ax.set_xlabel("Downlink tilt from nadir [deg]")
ax.grid(True, alpha=0.3)

plt.suptitle("Impact of Antenna Tilt on Network Performance\n(15 drones, K-means, urban)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side SINR heat maps: nadir vs tilted
fig, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)

for ax, tilt_deg, label in [(axes[0], 0, "Nadir (0° tilt)"), (axes[1], 30, "Tilted (30° tilt)")]:
    tilt_rad = np.full(base_scenario.drone_positions.shape[0], np.deg2rad(tilt_deg))

    # Compute SINR on a grid
    xs = np.linspace(0, 2000, 120)
    ys = np.linspace(0, 2000, 120)
    xx, yy = np.meshgrid(xs, ys)
    grid_pos = np.column_stack([xx.ravel(), yy.ravel(), np.zeros(xx.size)])

    pm = downlink_power_matrix(
        base_scenario.drone_positions, tilt_rad, base_scenario.dl_azimuth_rad,
        grid_pos, DOWNLINK_ANTENNA, channel, p_tx_w,
    )
    assoc = nearest_drone_association(base_scenario.drone_positions, grid_pos)
    sinr = compute_sinr(pm, assoc)
    sinr_map = sinr.sinr_db.reshape(120, 120)

    im = ax.pcolormesh(xx, yy, sinr_map, cmap="RdYlGn", shading="auto", vmin=-5, vmax=30)
    ax.scatter(base_scenario.drone_positions[:, 0], base_scenario.drone_positions[:, 1],
               c="black", s=60, marker="^", zorder=5)
    ax.set_xlabel("X [m]")
    ax.set_ylabel("Y [m]")
    ax.set_title(label)
    ax.set_aspect("equal")

fig.colorbar(im, ax=axes, label="SINR [dB]", shrink=0.8)
fig.suptitle("SINR Heat Map: Effect of Antenna Tilt", fontsize=13)
plt.show()

---
## 6. Altitude Sensitivity Analysis

How does drone altitude affect coverage? There's a tradeoff: higher altitude increases LoS probability but also increases path loss and broadens the coverage area (potentially more interference overlap).

In [ ]:
altitudes = np.arange(50, 350, 25)
alt_metrics = {"altitude": [], "coverage": [], "mean_sinr": [],
               "critical_sat": [], "throughput": []}

# Fixed user positions from the base scenario
user_pos = base_scenario.user_positions
user_tiers = base_scenario.user_tiers

for alt in altitudes:
    sc = create_scenario(
        n_drones=15, placement="kmeans", altitude_m=float(alt),
        n_clusters=5, users_per_cluster_mean=40,
        area_size_m=2000.0, critical_fraction=0.15, seed=42,
    )
    pm = downlink_power_matrix(
        sc.drone_positions, sc.dl_tilt_rad, sc.dl_azimuth_rad,
        sc.user_positions, DOWNLINK_ANTENNA, channel, p_tx_w,
    )
    assoc = nearest_drone_association(sc.drone_positions, sc.user_positions)
    sinr = compute_sinr(pm, assoc)
    m = compute_metrics(sinr, sc.user_tiers)

    alt_metrics["altitude"].append(alt)
    alt_metrics["coverage"].append(m.coverage_fraction)
    alt_metrics["mean_sinr"].append(m.mean_sinr_db)
    alt_metrics["critical_sat"].append(m.critical_sinr_satisfied)
    alt_metrics["throughput"].append(m.sum_throughput_mbps)

print("Altitude sweep complete.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.plot(alt_metrics["altitude"], alt_metrics["mean_sinr"], "o-", linewidth=2, color="#2c3e50")
ax.set_xlabel("Altitude [m]")
ax.set_ylabel("Mean SINR [dB]")
ax.set_title("Mean SINR vs Altitude")
ax.axvline(120, color="gray", linestyle=":", alpha=0.5, label="FAA Part 107 limit")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(alt_metrics["altitude"], alt_metrics["coverage"], "s-", linewidth=2, color="#27ae60")
ax.set_xlabel("Altitude [m]")
ax.set_ylabel("Coverage Fraction")
ax.set_title("Coverage vs Altitude")
ax.set_ylim(0, 1.05)
ax.axvline(120, color="gray", linestyle=":", alpha=0.5, label="FAA Part 107 limit")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(alt_metrics["altitude"], alt_metrics["critical_sat"], "D-", linewidth=2, color="#e74c3c")
ax.set_xlabel("Altitude [m]")
ax.set_ylabel("Critical Tier Satisfied")
ax.set_title("Critical Satisfaction vs Altitude")
ax.set_ylim(0, 1.05)
ax.axvline(120, color="gray", linestyle=":", alpha=0.5, label="FAA Part 107 limit")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle("Impact of Drone Altitude on Network Performance\n(15 drones, K-means, urban, nadir-pointing)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Summary Table

In [ ]:
print("=" * 80)
print("EXPERIMENT SUMMARY")
print("=" * 80)
print(f"{'Strategy':<12} {'Coverage':>10} {'Critical':>10} {'Mean SINR':>12} {'Throughput':>14}")
print("-" * 80)
for s in strategies:
    r = strategy_results[s]
    sm = r.summary()
    print(f"{s.capitalize():<12} "
          f"{sm['coverage_fraction']['mean']:>9.1%} "
          f"{sm['critical_sinr_satisfied']['mean']:>9.1%} "
          f"{sm['mean_sinr_db']['mean']:>10.1f} dB "
          f"{sm['sum_throughput_mbps']['mean']:>11.0f} Mbps")
print("-" * 80)
print(f"\nBest altitude for coverage (urban): "
      f"{alt_metrics['altitude'][np.argmax(alt_metrics['coverage'])]} m")
print(f"Best tilt for coverage:             "
      f"{tilt_metrics['tilt_deg'][np.argmax(tilt_metrics['coverage'])]}\u00b0")
print("\nKey takeaway: antenna orientation and placement strategy both")
print("significantly impact coverage and interference -- motivating the")
print("geometry-aware joint optimization proposed in this thesis.")